Url Connection & Data Cleaning

In [122]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 3

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
  master_df = pd.concat(dataframe_list, ignore_index=True)
  print("\n Collection complete!")
  print(f"Total rows collected across all locations: {len(master_df)}")
else:
  print("\n No data was collected. Check your API key or coordinates.")
master_df.head()


Successfully added 45 fire points from Amazon Rainforest.
Successfully added 130 fire points from California.
Successfully added 1566 fire points from Siberian Taiga.
Successfully added 106 fire points from New South Wales.
Successfully added 1065 fire points from Pantanal.
Successfully added 78 fire points from Alberta.
Successfully added 1543 fire points from Mediterranean Basin.
Successfully added 886 fire points from Indonesia.
Successfully added 68 fire points from Greece.
Successfully added 710 fire points from Algeria.

 Collection complete!
Total rows collected across all locations: 6197


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,location_name
0,0.00536,-78.10262,297.14,0.47,0.39,2026-05-29,652,N20,VIIRS,n,2.0NRT,278.25,0.90,N,Amazon Rainforest
1,3.66483,-74.88856,298.24,0.43,0.46,2026-05-29,652,N20,VIIRS,n,2.0NRT,275.51,0.53,N,Amazon Rainforest
2,-4.42141,-54.99192,333.16,0.44,0.62,2026-05-29,1752,N20,VIIRS,n,2.0NRT,288.83,2.67,D,Amazon Rainforest
3,-4.34595,-56.30996,345.53,0.36,0.57,2026-05-29,1752,N20,VIIRS,n,2.0NRT,289.51,7.80,D,Amazon Rainforest
4,-4.34557,-56.30969,339.55,0.36,0.57,2026-05-29,1752,N20,VIIRS,n,2.0NRT,289.52,5.40,D,Amazon Rainforest


Data Cleaning

In [123]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [124]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                          frp             ...  bright_ti4           
                         mean        std  ...        mean        std
location_name                             ...                       
Alberta              9.401923  16.414235  ...  332.179359  19.338892
Algeria              3.134296   7.238049  ...  316.576620  15.248840
Amazon Rainforest    4.615778   2.696746  ...  332.426000  14.100266
California           3.008923   6.329351  ...  315.440462  18.087911
Greece               1.605588   1.446989  ...  309.664559  15.626472
Indonesia            8.189278  15.710582  ...  330.301490  16.912625
Mediterranean Basin  3.527686   6.838330  ...  319.360285  17.859600
New South Wales      3.210000   4.066914  ...  315.585566  17.598533
Pantanal             3.075399   3.577374  ...  320.228347  16.010185
Siberian Taiga       8.304700  13.640374  ...  327.498978  19.825622

[10 rows x 6 columns]


    frp  bright_ti5  bright_ti4  track  scan
0  0.90      278.25      297.14

Creating Risk Level Column

In [125]:
def assign_risk_level(frp_value):
  if frp_value < 2.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)
master_df

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,location_name,risk_level
0,0.00536,-78.10262,297.14,0.47,0.39,2026-05-29,652,N20,VIIRS,n,2.0NRT,278.25,0.90,N,Amazon Rainforest,0
1,3.66483,-74.88856,298.24,0.43,0.46,2026-05-29,652,N20,VIIRS,n,2.0NRT,275.51,0.53,N,Amazon Rainforest,0
2,-4.42141,-54.99192,333.16,0.44,0.62,2026-05-29,1752,N20,VIIRS,n,2.0NRT,288.83,2.67,D,Amazon Rainforest,1
3,-4.34595,-56.30996,345.53,0.36,0.57,2026-05-29,1752,N20,VIIRS,n,2.0NRT,289.51,7.80,D,Amazon Rainforest,1
4,-4.34557,-56.30969,339.55,0.36,0.57,2026-05-29,1752,N20,VIIRS,n,2.0NRT,289.52,5.40,D,Amazon Rainforest,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6192,35.56804,-0.93753,330.52,0.38,0.58,2026-05-31,1221,N20,VIIRS,l,2.0NRT,305.25,1.08,D,Algeria,0
6193,35.69154,2.72518,340.63,0.47,0.48,2026-05-31,1221,N20,VIIRS,n,2.0NRT,310.39,3.18,D,Algeria,1
6194,35.80827,-0.25258,331.07,0.34,0.56,2026-05-31,1221,N20,VIIRS,n,2.0NRT,301.02,2.24,D,Algeria,1
6195,35.87790,4.44963,341.33,0.38,0.43,2026-05-31,1221,N20,VIIRS,n,2.0NRT,315.19,2.48,D,Algeria,1


Prediction: Fire Radiative Power & Bright Fire Spots

In [126]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")


# Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=120, criterion="log_loss", max_samples=0.8, random_state=42)
rf_model.fit(X_train, y_train, sample_weight=None)
predictions = rf_model.predict(X_test)

# Add a new Dataframe to see the results clearly
test_results_df = X_test.copy()
test_results_df['True_Risk_Level'] = y_test
test_results_df['Predicted_Risk_level'] = predictions

test_results_df.head(50)

X_train: (4957, 4), X_test: (1240, 4), y_train: (4957,), y_test: (1240,)


,track,scan,bright_ti5,bright_ti4,True_Risk_Level,Predicted_Risk_level
5040,0.40,0.47,281.87,327.21,1,1
4781,0.37,0.41,292.81,337.12,1,1
1203,0.76,0.73,275.50,298.99,0,0
5185,0.47,0.46,293.98,336.40,1,1
2316,0.37,0.41,299.71,333.39,1,1
3065,0.38,0.43,291.22,312.36,1,1
4885,0.60,0.40,290.96,356.18,1,1
3206,0.45,0.40,289.70,342.22,1,1
132,0.36,0.39,311.32,338.92,1,1
4517,0.37,0.40,310.93,340.23,1,1


Accuracy Score: How good were the model's guesses?

In [127]:
from sklearn.metrics import accuracy_score as acc
from sklearn.metrics import precision_score as pre
from sklearn.metrics import recall_score as rc
from sklearn.metrics import f1_score as f1
from sklearn.metrics import classification_report as cl_report
acc_score = acc(test_results_df['True_Risk_Level'], test_results_df['Predicted_Risk_level'])
pre_score = pre(test_results_df['True_Risk_Level'], test_results_df['Predicted_Risk_level'], average='weighted')
rec_score = rc(test_results_df['True_Risk_Level'], test_results_df['Predicted_Risk_level'],average='weighted')
f1_score = f1(test_results_df['True_Risk_Level'], test_results_df['Predicted_Risk_level'], average='weighted')
print(f" Accuracy Score: {acc_score}\n Precision Score: {pre_score}\n Recall Score: {rec_score}\n F1 Score: {f1_score}")
print("\n")
print(cl_report(
    test_results_df['True_Risk_Level'], 
    test_results_df['Predicted_Risk_level'], 
    target_names=['Low Risk (0)', 'Moderate Risk (1)', 'High Risk (2)', 'Extreme Risk (3)']
))


 Accuracy Score: 0.8491935483870968
 Precision Score: 0.8450181254975718
 Recall Score: 0.8491935483870968
 F1 Score: 0.8369095156711025


                   precision    recall  f1-score   support

     Low Risk (0)       0.88      0.89      0.89       529
Moderate Risk (1)       0.83      0.90      0.86       629
    High Risk (2)       0.69      0.19      0.30        57
 Extreme Risk (3)       0.88      0.28      0.42        25

         accuracy                           0.85      1240
        macro avg       0.82      0.57      0.62      1240
     weighted avg       0.85      0.85      0.84      1240

